# 28 - Known-ground-truth localization benchmark (three datasets)

Revision experiment for the Computer Networks submission. It answers the main
reviewer objection: the model-free anchor GT_in (per-feature Wasserstein-1)
is itself a marginal statistic, so scoring KS against it favours KS. Here the
drifted feature set S is planted, so the answer key is known exactly and
belongs to no localizer family.

Three drift arms, all built on real z-scored flow vectors of each dataset:
  covariate   : a fraction q of post rows get +DELTA on S (marginals of S move).
  dependency  : a fraction q of post rows get their S-block copied, as a unit,
                from another random row. Every marginal and the joint law of S
                are preserved; only S's dependence on the remaining features
                changes (the Kulinski et al. 2020 "feature shift" setting).
  concept     : P(X) fixed; the synthetic label rule moves from S1 to S2 = S
                (the 07e construction, now on all three datasets).

Localizers: KS, cond-KS (KS on linear-regression residuals, a light stand-in
for conditional feature-shift tests), domain-SHAP, perm-domain, aux-SHAP,
dual, random. Models RF / XGB / MLP. Seeds {42,1,7,13,99}.

PRE-REGISTRATION: reports/advisor_notes/PREREG_known_gt.md must be committed to git;
record that commit hash in the cell below BEFORE running any cell after it.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/phd_thesis')
%pip install -q shap xgboost

In [3]:
import time, json, datetime, os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, wilcoxon
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

from src.utils.seeding import set_global_seed
from src.data.loaders import load_dataset, CSE2018_DAY_ORDER
from src.divergence.localization import (
    ks_drift_scores, topk_indices, precision_at_k, random_precision_expectation,
    shap_importance_any,
)

ROOT = Path('/content/drive/MyDrive/phd_thesis')
TABLES = ROOT / 'reports' / 'tables'
PROCESSED = ROOT / 'data' / 'processed'
RAW_CSV = TABLES / '28_known_gt_raw.csv'          # checkpoint, appended per run
SUMMARY_JSON = PROCESSED / 'phase8_28_known_gt.json'
TABLES.mkdir(parents=True, exist_ok=True)

PREREG_GIT_HASH = '431365ba849254f20c5a942c11e910bbdacf63b2'         # hash of the PREREG commit
assert PREREG_GIT_HASH != 'FILL-ME-BEFORE-RUNNING', 'commit PREREG_known_gt.md first'

with open(PROCESSED / 'phase1_06_metadata.json') as f:
    RF_PARAMS = json.load(f)['rf_hyperparameters']
MLP_PARAMS = dict(hidden_layer_sizes=(256, 128, 64), activation='relu',
                  alpha=1e-4, max_iter=300, early_stopping=True,
                  n_iter_no_change=15, random_state=42)

In [4]:
# ---- fixed design (mirrors PREREG_known_gt.md; do not edit after commit) ----
DATASETS = ['cicids2017', 'unsw_nb15', 'cse_cic_ids2018']
POOL_N = 40_000            # real rows drawn per dataset
WINDOW = 4000              # rows per pre / post window
GT_SIZE = 10               # planted |S|
K = 10                     # precision@10, so chance = GT_SIZE / d
DELTA = 1.0                # covariate shift size, z-units
Q_GRID = [0.10, 0.25, 0.50]  # fraction of post rows affected (covariate, dependency)
MODAL_MAX = 0.90           # non-degenerate: modal value share <= 0.90 (07h rule)
SEEDS = [42, 1, 7, 13, 99]
MODELS = ['rf', 'xgb', 'mlp']
SHAP_SAMPLE = 800
PERM_EVAL = 1200
PERM_REPEATS = 3
LABEL_FREE = ['ks', 'cond_ks', 'random']
MODEL_BASED = ['domain_shap', 'perm_domain', 'aux_shap', 'dual']
print(f'chance per dataset is GT_SIZE/d; seeds {SEEDS}; models {MODELS}')

chance per dataset is GT_SIZE/d; seeds [42, 1, 7, 13, 99]; models ['rf', 'xgb', 'mlp']


In [5]:
def load_pool(name, seed=42):
    """Real feature vectors for one dataset: numeric, non-constant, z-scored."""
    rng = np.random.default_rng(seed)
    if name == 'cse_cic_ids2018':
        parts = []
        for day in CSE2018_DAY_ORDER:
            Xd, _, _ = load_dataset(name, day=day)
            idx = rng.choice(len(Xd), size=min(POOL_N // len(CSE2018_DAY_ORDER), len(Xd)),
                             replace=False)
            parts.append(Xd.iloc[idx])
            del Xd
        X = pd.concat(parts, ignore_index=True)
    else:
        X, _, _ = load_dataset(name)
        X = X.iloc[rng.choice(len(X), size=min(POOL_N, len(X)), replace=False)]
    X = X.select_dtypes(include=[np.number]).astype(float)
    X = X.loc[:, X.std() > 0]
    Xv = X.to_numpy()
    Xv = (Xv - Xv.mean(0)) / Xv.std(0)
    modal = np.array([pd.Series(Xv[:, j]).value_counts(normalize=True).iloc[0]
                      for j in range(Xv.shape[1])])
    nondegen = np.flatnonzero(modal <= MODAL_MAX)
    return Xv.astype(np.float32), list(X.columns), nondegen


def make_model(kind, seed):
    if kind == 'rf':
        return RandomForestClassifier(**{**RF_PARAMS, 'random_state': seed})
    if kind == 'mlp':
        return MLPClassifier(**{**MLP_PARAMS, 'random_state': seed})
    from xgboost import XGBClassifier
    return XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                         subsample=0.9, colsample_bytree=0.9, n_jobs=-1,
                         tree_method='hist', random_state=seed, eval_metric='logloss')


def norm(v):
    v = np.abs(np.asarray(v, float)); s = v.sum(); return v / s if s > 0 else v


def imp(model, X_eval, X_bg, seed):
    return norm(shap_importance_any(model, X_eval, X_bg=X_bg, seed=seed))


def cond_ks_scores(X_pre, X_post, seed, alpha=1.0):
    """Per-feature KS on residuals of a ridge regression of x_j on x_{-j},
    fitted on half of the pre window. Sensitive to changes in a feature's
    dependence on the others, which a marginal KS cannot see."""
    rng = np.random.default_rng(seed)
    fit = rng.random(len(X_pre)) < 0.5
    A, B = X_pre[fit], X_pre[~fit]
    d = X_pre.shape[1]
    out = np.zeros(d)
    for j in range(d):
        rest = np.delete(np.arange(d), j)
        r = Ridge(alpha=alpha).fit(A[:, rest], A[:, j])
        res_ref = B[:, j] - r.predict(B[:, rest])
        res_cur = X_post[:, j] - r.predict(X_post[:, rest])
        out[j] = ks_2samp(res_ref, res_cur).statistic
    return out


def synth_labels(X, S, rng):
    w = np.zeros(X.shape[1]); w[S] = rng.normal(size=len(S))
    s = X @ w
    return (s > np.median(s)).astype(int), w


def build_arm(POOL, nondegen, arm, q, seed):
    """Return X_pre, y_pre, X_post, y_post, S (planted drifted set)."""
    rng = np.random.default_rng(seed)
    rows = rng.permutation(len(POOL))
    X_pre = POOL[rows[:WINDOW]].copy()
    X_post = POOL[rows[WINDOW:2 * WINDOW]].copy()
    S = rng.choice(nondegen, size=GT_SIZE, replace=False)
    if arm == 'concept':
        others = np.setdiff1d(nondegen, S)
        S1 = rng.choice(others, size=GT_SIZE, replace=False)
        y_pre, _ = synth_labels(X_pre, S1, rng)
        y_post, _ = synth_labels(X_post, S, rng)
        return X_pre, y_pre, X_post, y_post, S
    # covariate / dependency: a fixed label rule on a set R disjoint from S, so
    # detectors (needed for aux/dual) exist but label relevance cannot hand any
    # localizer the answer; the drift itself is in P(X) only
    R = rng.choice(np.setdiff1d(nondegen, S), size=GT_SIZE, replace=False)
    y_pre, w = synth_labels(X_pre, R, rng)
    hit = rng.random(WINDOW) < q
    if arm == 'covariate':
        X_post[np.ix_(hit, S)] += DELTA
    elif arm == 'dependency':
        donors = rng.choice(len(POOL), size=int(hit.sum()), replace=False)
        X_post[np.ix_(hit, S)] = POOL[np.ix_(donors, S)]
    s = X_post @ w
    y_post = (s > np.median(X_pre @ w)).astype(int)
    return X_pre, y_pre, X_post, y_post, S


def localize(X_pre, y_pre, X_post, y_post, model, seed):
    rng = np.random.default_rng(seed)
    d = X_pre.shape[1]
    scores = {'ks': ks_drift_scores(X_pre, X_post),
              'cond_ks': cond_ks_scores(X_pre, X_post, seed),
              'random': rng.random(d)}
    domX = np.vstack([X_pre, X_post])
    domy = np.r_[np.zeros(len(X_pre), int), np.ones(len(X_post), int)]
    dtr, dev = train_test_split(np.arange(len(domy)), train_size=0.7,
                                stratify=domy, random_state=seed)
    dom = make_model(model, seed).fit(domX[dtr], domy[dtr])
    di = rng.choice(dev, size=min(SHAP_SAMPLE, len(dev)), replace=False)
    scores['domain_shap'] = imp(dom, domX[di], domX[dtr], seed)
    pe = rng.choice(dev, size=min(PERM_EVAL, len(dev)), replace=False)
    scores['perm_domain'] = norm(np.clip(permutation_importance(
        dom, domX[pe], domy[pe], n_repeats=PERM_REPEATS,
        random_state=seed).importances_mean, 0, None))
    dom_auc = float(dom.score(domX[dev], domy[dev]))
    f_pre = make_model(model, seed).fit(X_pre, y_pre)
    f_post = make_model(model, seed).fit(X_post, y_post)
    sh = X_post[rng.choice(len(X_post), size=min(SHAP_SAMPLE, len(X_post)), replace=False)]
    a_post = imp(f_post, sh, X_post, seed)
    a_pre = imp(f_pre, sh, X_pre, seed)
    scores['aux_shap'] = a_post
    scores['dual'] = np.abs(a_post - a_pre)
    return scores, dom_auc

In [6]:
# ---- run grid, checkpointing every condition so a Colab disconnect resumes ----
done = set()
if RAW_CSV.exists():
    prev = pd.read_csv(RAW_CSV)
    done = set(zip(prev.dataset, prev.arm, prev.q, prev.model, prev.seed))
    print(f'resuming: {len(done)} conditions already in {RAW_CSV.name}')

conds = [('covariate', q) for q in Q_GRID] + [('dependency', q) for q in Q_GRID] + [('concept', 1.0)]
t0 = time.time()
geometry = {}
for ds in DATASETS:
    set_global_seed(42)
    POOL, cols, nondegen = load_pool(ds)
    d = POOL.shape[1]
    geometry[ds] = {'d': d, 'nondegenerate': int(len(nondegen)),
                    'chance': random_precision_expectation(GT_SIZE, d)}
    print(f'{ds}: pool {POOL.shape}, non-degenerate {len(nondegen)}, chance {GT_SIZE/d:.3f}')
    for arm, q in conds:
        for seed in SEEDS:
            X_pre, y_pre, X_post, y_post, S = build_arm(POOL, nondegen, arm, q, seed)
            ks_max = float(ks_drift_scores(X_pre, X_post).max())
            for model in MODELS:
                if (ds, arm, q, model, seed) in done:
                    continue
                scores, dom_acc = localize(X_pre, y_pre, X_post, y_post, model, seed)
                out = [{'dataset': ds, 'arm': arm, 'q': q, 'model': model, 'seed': seed,
                        'method': m, 'precision': precision_at_k(sc, S, K),
                        'chance': GT_SIZE / d, 'ks_max': ks_max,
                        'domain_acc': dom_acc} for m, sc in scores.items()]
                pd.DataFrame(out).to_csv(RAW_CSV, mode='a', index=False,
                                         header=not RAW_CSV.exists())
            print(f'  {ds} {arm} q={q} seed={seed}  ({time.time()-t0:.0f}s)')

cicids2017: pool (40000, 81), non-degenerate 70, chance 0.123


/usr/local/lib/python3.13/dist-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
/usr/local/lib/python3.13/dist-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
/usr/local/lib/python3.13/dist-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)


  cicids2017 covariate q=0.1 seed=42  (183s)


/usr/local/lib/python3.13/dist-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
/usr/local/lib/python3.13/dist-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
/usr/local/lib/python3.13/dist-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)


  cicids2017 covariate q=0.1 seed=1  (341s)


...

  cse_cic_ids2018 concept q=1.0 seed=7  (13563s)


/usr/local/lib/python3.13/dist-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
/usr/local/lib/python3.13/dist-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
/usr/local/lib/python3.13/dist-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)


  cse_cic_ids2018 concept q=1.0 seed=13  (13710s)


/usr/local/lib/python3.13/dist-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
/usr/local/lib/python3.13/dist-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
/usr/local/lib/python3.13/dist-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)


  cse_cic_ids2018 concept q=1.0 seed=99  (13859s)


In [7]:
# ---- summary and pre-registered paired tests (Holm across the family) ----
res = pd.read_csv(RAW_CSV)
summary = (res.groupby(['dataset', 'arm', 'q', 'method'])['precision']
              .mean().unstack('method').round(3))
print(summary.to_string())
summary.to_csv(TABLES / '28_known_gt_summary.csv')

def paired(focal, ref, q_filter):
    a = res[q_filter & (res.method == focal)].set_index(['dataset', 'q', 'model', 'seed'])['precision']
    b = res[q_filter & (res.method == ref)].set_index(['dataset', 'q', 'model', 'seed'])['precision']
    a, b = a.align(b, join='inner')
    diff = (a - b).to_numpy()
    rng = np.random.default_rng(42)
    boots = [rng.choice(diff, len(diff)).mean() for _ in range(10_000)]
    p = wilcoxon(diff).pvalue if np.any(diff != 0) else 1.0
    sd = diff.std(ddof=1)
    return {'focal': focal, 'ref': ref, 'n': len(diff), 'delta': diff.mean(),
            'ci_lo': np.percentile(boots, 2.5), 'ci_hi': np.percentile(boots, 97.5),
            'p': p, 'dz': diff.mean() / sd if sd > 0 else np.nan}

TESTS = [  # (label, focal, ref, arm, trees_only)  -- fixed in PREREG section 4
    ('P1 covariate: KS vs domain-SHAP (trees)', 'ks', 'domain_shap', 'covariate', True),
    ('P2 dependency: domain-SHAP vs KS', 'domain_shap', 'ks', 'dependency', False),
    ('P3 dependency: cond-KS vs KS', 'cond_ks', 'ks', 'dependency', False),
    ('P4 concept: aux-SHAP vs KS', 'aux_shap', 'ks', 'concept', False),
    ('P5 all arms: dual vs aux-SHAP', 'dual', 'aux_shap', None, False),
    ('P6 covariate: KS vs random', 'ks', 'random', 'covariate', False),
]
rows = []
for label, f, r, arm, trees in TESTS:
    filt = pd.Series(True, index=res.index)
    if arm is not None:
        filt &= res.arm == arm
    if trees:
        filt &= res.model.isin(['rf', 'xgb'])
    rows.append({'test': label, **paired(f, r, filt)})
tests = pd.DataFrame(rows)
order = np.argsort(tests.p.values)
m = len(tests); adj = np.empty(m); running = 0.0
for rank, i in enumerate(order):
    running = max(running, min(1.0, (m - rank) * tests.p.values[i]))
    adj[i] = running
tests['p_holm'] = adj
print(tests.round(4).to_string())
tests.to_csv(TABLES / '28_known_gt_tests.csv', index=False)

SUMMARY = {'generated': datetime.datetime.now().isoformat(timespec='seconds'),
           'prereg_git_hash': PREREG_GIT_HASH, 'geometry': geometry,
           'design': {'gt_size': GT_SIZE, 'k': K, 'delta': DELTA, 'q_grid': Q_GRID,
                      'seeds': SEEDS, 'models': MODELS, 'window': WINDOW},
           'tests': tests.to_dict(orient='records')}
with open(SUMMARY_JSON, 'w') as f:
    json.dump(SUMMARY, f, indent=2, default=float)
print(f'saved {SUMMARY_JSON.name}; send 28_known_gt_raw.csv, _summary.csv, _tests.csv back')

method                           aux_shap  cond_ks  domain_shap   dual    ks  perm_domain  random
dataset         arm        q                                                                     
cicids2017      concept    1.00     0.307     0.22        0.207  0.247  0.14        0.133    0.06
                covariate  0.10     0.080     0.20        0.667  0.193  1.00        0.180    0.06
                           0.25     0.047     0.42        0.753  0.253  1.00        0.333    0.06
                           0.50     0.047     0.50        0.847  0.307  1.00        0.560    0.06
                dependency 0.10     0.033     0.32        0.427  0.233  0.10        0.273    0.06
                           0.25     0.013     0.28        0.487  0.260  0.10        0.567    0.06
                           0.50     0.000     0.32        0.587  0.273  0.14        0.633    0.06
cse_cic_ids2018 concept    1.00     0.360     0.14        0.127  0.273  0.22        0.200    0.02
                cova